In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel, pybel
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import shutil
import subprocess
import csv
import time
import traceback
import sys
import shutil

In [ ]:




os.environ['PATH'] = '/home/fwtop/apps/openmpi/bin:' + os.environ.get('PATH', '')

os.environ['LD_LIBRARY_PATH'] = '/home/fwtop/apps/openmpi/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

In [ ]:

MAX_TASKS = 1

In [ ]:

NPROC = 20
MEM =4000  

In [ ]:
directory = "opt+freq"

os.makedirs("opt+freq/imaginary_frequency", exist_ok=True)
imaginary_frequency_path = os.path.join(directory, "imaginary_frequency")
current_path=os.getcwd()
failure_path = os.path.join(current_path,directory, "failure")
success_path = os.path.join(directory, "success")

In [ ]:
import os
import re
import shutil

def check_for_imaginary_frequencies(opt_success_path, imaginary_frequency_path):
    
    
    
    
    
    freq_pattern = re.compile(r'^\s*\d+:\s+(-?\d+\.\d+)\s+cm\*\*-1', re.M)
    
    imaginary_freq_count = 0  
    total_files = 0  

    
    for filename in os.listdir(opt_success_path):
        if filename.endswith(".out"):
            file_path = os.path.join(opt_success_path, filename)
            total_files += 1  
            
            with open(file_path, 'r') as file:
                
                lines = file.readlines()
                start_index = None  
                
                
                for idx in range(len(lines) - 1, -1, -1):
                    if "VIBRATIONAL FREQUENCIES" in lines[idx]:
                        start_index = idx
                        break
                
                
                if start_index is not None:
                    relevant_text = ''.join(lines[start_index:])
                else:
                    relevant_text = ''.join(lines)
                
                
                frequencies = freq_pattern.findall(relevant_text)
                
                
                has_imaginary_freq = any(float(freq) < 0 for freq in frequencies)
                
                if has_imaginary_freq:
                    base_filename = os.path.splitext(filename)[0]
                    imaginary_freq_count += 1  
                    
                    for ext in ['.inp', '.out', '.opt', '.xyz', '.txt', '.gbw', '.engrad', 
                                '.densitiesinfo', '.densities', '.hess', '.bibtex']:
                        file_to_move = base_filename + ext
                        src_file_path = os.path.join(opt_success_path, file_to_move)
                        if os.path.exists(src_file_path):
                            shutil.move(src_file_path, imaginary_frequency_path)
                            print(f"Moved {file_to_move} to imaginary frequency folder.")

    
    imaginary_freq_rate = (imaginary_freq_count / total_files) * 100 if total_files > 0 else 0
    
    print("Finished checking for imaginary frequencies.")
    print(f"Number of imaginary_freq calculations: {imaginary_freq_count} ({imaginary_freq_rate:.2f}%)")
    print(f"Total number of files: {total_files}")

In [ ]:


def extract_opt_coordinates(xyz_file):
    with open(xyz_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    coordinates_lines = lines[start_index : final_index]
    
    
    coordinates_str = ''.join(coordinates_lines)
    
    return coordinates_str

In [ ]:

def extract_inp_charge_and_multiplicity_line(inp_file):
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index - 1]        
    
    return charge_and_multiplicity_line

In [ ]:

def create_opt_for_imaginary_frequency_correction_inp(opt_xyz_file, opt_inp_file, output_opt_inp_file, opt_gbw_file, mem, nproc):
    
    
    coordinates_str = extract_opt_coordinates(opt_xyz_file)
    
    charge_and_multiplicity_line = extract_inp_charge_and_multiplicity_line(opt_inp_file)
    
    
    with open(output_opt_inp_file, 'w') as f:
        
        f.write("! B3LYP D3 def2-TZVP def2/J RIJCOSX opt freq tightSCF noautostart miniprint nofinalgridx\n") 
        
        f.write(f"%maxcore     {mem}\n")
        
        f.write(f"%pal nprocs  {nproc} end\n")
        
        f.write("%geom calc_Hess true Recalc_Hess  10 end\n") 
        
        f.write(f"%scf MOInp \"{opt_gbw_file}\"  AutoStart false end\n") 
        
        f.write(charge_and_multiplicity_line)
        
        f.write(coordinates_str)
        
        f.write("*")

In [ ]:

def generate_total_imaginary_frequency_inp(opt_freq_path, imaginary_frequency_file_path, mem=2000, nproc=20):
    
    
    if not os.path.exists(opt_freq_path):
        os.makedirs(opt_freq_path)
    
    
    for filename in os.listdir(imaginary_frequency_file_path):
        if filename.endswith(".out"):
            
            base_name = os.path.splitext(filename)[0]
            
            opt_xyz_file = os.path.join(imaginary_frequency_file_path, base_name + ".xyz")
            
            opt_inp_file = os.path.join(imaginary_frequency_file_path, base_name + ".inp")
            
            opt_gbw_file = os.path.abspath(os.path.join(imaginary_frequency_file_path, base_name + ".gbw"))
            
            
            output_opt_inp_file = os.path.join(opt_freq_path, base_name + ".inp")
            
            
            create_opt_for_imaginary_frequency_correction_inp(opt_xyz_file, opt_inp_file, output_opt_inp_file, opt_gbw_file, mem, nproc)
            
            
            print(f"Processed {base_name}: Generated {output_opt_inp_file}")

In [ ]:
def run_orca_gas_optfreq(file, success_dir, failure_dir):
    
    
    base_name = os.path.splitext(file)[0]
    output_file = base_name + '.out'
    opt_file = base_name + '.opt'
    trj_file = base_name + '_trj' + '.xyz'
    xyz_file = base_name + '.xyz'
    property_file = base_name + '.property' + '.txt'
    gbw_file = base_name + '.gbw'
    engrad_file = base_name + '.engrad'
    densitiesinfo_file = base_name + '.densitiesinfo'
    densities_file = base_name + '.densities'
    hess_file = base_name + '.hess'
    bibtex_file = base_name + '.bibtex'

    try:
        
        with open(output_file, 'w') as out:
            subprocess.run(['/home/public/orca_6_0_1_linux_x86-64_shared_openmpi416_avx2/orca', file], stdout=out, check=True)

        
        with open(output_file, 'r') as f:
            content = f.read()
            if 'ORCA TERMINATED NORMALLY' in content:
                
                files_to_move = [
                    file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(success_dir, os.path.basename(f)))
                
                new_output_path = os.path.join(success_dir, os.path.basename(output_file))
                return new_output_path, True
            else:
                
                files_to_move = [
                    file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(failure_dir, os.path.basename(f)))
                return None, False

    except subprocess.CalledProcessError as e:
        
        os.rename(file, os.path.join(failure_dir, os.path.basename(file)))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(failure_dir, os.path.basename(output_file)))
        print(f"Error running ORCA for file {file}: {str(e)}")
        print(traceback.format_exc())
        return None, False

In [ ]:

opt_freq_path = "opt+freq"
imaginary_frequency_file_path = os.path.join(opt_freq_path, "imaginary_frequency")
os.makedirs("opt+freq/imaginary_frequency", exist_ok=True)
success_dir = os.path.join(opt_freq_path, 'success')
failure_dir = os.path.join(opt_freq_path, 'failure')

inp_files = [
    os.path.join(opt_freq_path, f) for f in os.listdir(opt_freq_path) if f.endswith(".inp")
]


start_time = time.time() 


check_for_imaginary_frequencies(success_path, imaginary_frequency_path)


generate_total_imaginary_frequency_inp(opt_freq_path, imaginary_frequency_file_path, mem=MEM, nproc=NPROC)


with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
    
    futures = [executor.submit(run_orca_gas_optfreq, file, success_dir, failure_dir) for file in inp_files] 
    
    results = [f.result() for f in futures]


check_for_imaginary_frequencies(success_dir, imaginary_frequency_file_path)


end_time = time.time()


elapsed_time = end_time - start_time
print(f"The code ran for {elapsed_time} seconds.")